In [1]:
!pip install nflreadpy

  Using cached nflreadpy-0.1.5-py3-none-any.whl.metadata (7.5 kB)
  Using cached polars-1.39.3-py3-none-any.whl.metadata (10 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached polars_runtime_32-1.39.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached nflreadpy-0.1.5-py3-none-any.whl (30 kB)
Using cached polars-1.39.3-py3-none-any.whl (823 kB)
Using cached polars_runtime_32-1.39.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (46.9 MB)
Using cached pydantic_settings-2.13.1-py3-none-any.whl (58 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)


In [2]:
import nflreadpy as nfl
import pandas as pd

player_stats = nfl.load_player_stats([2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025])

stats_df = player_stats.to_pandas()

In [33]:
stats_df.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,...,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr,game_id
0,00-0000108,D.Akers,David Akers,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2012,1,REG,SF,...,0,1.0,0,0,0,0,0,0.0,0.0,None
1,00-0000585,C.Bailey,Champ Bailey,CB,DB,https://static.www.nfl.com/image/private/f_aut...,2012,1,REG,DEN,...,0,NaN,0,0,0,0,0,0.0,0.0,None
2,00-0000741,R.Barber,Ronde Barber,FS,DB,https://static.www.nfl.com/image/private/f_aut...,2012,1,REG,TB,...,0,NaN,0,0,0,0,0,0.0,0.0,None
3,00-0001820,K.Brooking,Keith Brooking,MLB,LB,https://static.www.nfl.com/image/private/f_aut...,2012,1,REG,DEN,...,0,NaN,0,0,0,0,0,0.0,0.0,None
4,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2012,1,REG,CLE,...,0,1.0,0,0,0,0,0,0.0,0.0,None


In [4]:
stats_df.columns.values

array(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'season', 'week', 'season_type',
       'team', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'passing_interceptions',
       'sacks_suffered', 'sack_yards_lost', 'sack_fumbles',
       'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'carries',
       'rushing_yards', 'rushing_tds', 'rushing_fumbles',
       'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa',
       'rushing_2pt_conversions', 'receptions', 'targets',
       'receiving_yards', 'receiving_tds', 'receiving_fumbles',
       'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_epa', 'receiving_2pt_conversions', 'racr',
       'target_share', 'air_yards_share', 'wopr', '

In [5]:
stat_list = {
    "qb": ["season", "week", "passing_first_downs", "completions", "attempts",
           "passing_yards", "passing_tds", "passing_interceptions",
           "sacks_suffered", "rushing_yards", "rushing_tds"],
    "wr": ["season", "week", "receptions", "targets", "receiving_yards",
           "receiving_tds", "receiving_fumbles", "receiving_fumbles_lost",
           "receiving_first_downs", "receiving_air_yards",
           "receiving_yards_after_catch"],
    "rb": ["season", "week", "carries", "rushing_yards", "rushing_tds",
           "rushing_fumbles", "rushing_fumbles_lost", "rushing_first_downs",
           "receptions", "targets", "receiving_yards", "receiving_tds",
           "receiving_first_downs", "rushing_2pt_conversions"],
    "lb": ["season", "week", "def_tackles_solo", "def_tackles_with_assist",
           "def_tackle_assists", "def_tackles_for_loss",
           "def_tackles_for_loss_yards", "def_fumbles_forced",
           "def_sacks", "def_sack_yards", "def_qb_hits",
           "def_interceptions", "def_interception_yards",
           "def_pass_defended", "def_tds", "def_fumbles"],
    "cb": ["season", "week", "def_tackles_solo", "def_tackles_with_assist",
           "def_tackle_assists", "def_interceptions",
           "def_interception_yards", "def_pass_defended",
           "def_fumbles_forced", "def_fumbles", "def_tds",
           "def_tackles_for_loss", "def_sacks", "def_qb_hits"]
}

In [6]:
def career_averages(df, stat_cols):
    """
    Gets a players career average for a variety of stats based on their positions

    Args:
        df: DataFrame of a single player's game-by-game stats
        stat_cols: List of column names to average

    Returns:
        Single-row DataFrame with career averages and a games_played column
    """
    valid_cols = [col for col in stat_cols if col in df.columns]

    numeric_df = df[valid_cols].apply(pd.to_numeric, errors='coerce')

    averages = numeric_df.mean().to_frame().T
    
    for col in ['player_id', 'player_display_name', 'position']:
        if col in df.columns:
            averages.insert(0, col, df[col].iloc[0])

    averages.insert(len(averages.columns) - len(valid_cols), 'games_played', len(df))

    return averages.reset_index(drop=True)

In [8]:
def get_data_on(player_name, position):
    player_df = stats_df[stats_df["player_display_name"] == player_name]
    stat_cols = [col for col in stat_list[position] if col not in ['season', 'week']]
    avgs = career_averages(player_df, stat_cols)

    parts = player_name.split()
    first_initial = parts[0][0]
    last_name = parts[-1]
    
    avgs.to_csv(f'data/NFL/{position.upper()}/{first_initial}{last_name}_{position.upper()}.csv', index=False)
    print(avgs)

In [32]:
get_data_on("Trevon Diggs", "cb")

  position player_display_name   player_id  games_played  def_tackles_solo  \
0       CB        Trevon Diggs  00-0036361            71          2.732394   

   def_tackles_with_assist  def_tackle_assists  def_interceptions  \
0                 0.197183            0.633803            0.28169   

   def_interception_yards  def_pass_defended  def_fumbles_forced  def_fumbles  \
0                2.873239           0.887324            0.028169          0.0   

    def_tds  def_tackles_for_loss  def_sacks  def_qb_hits  
0  0.028169              0.028169   0.014085          0.0  
